[← GstreamerExp hub](../../index.html) · [README](../../README.md) · [Hypothesis catalog](../../docs/HYPOTHESES.md)

# H7 — With the encoder starved, the queue-delay-target knob starts trading picture quality

**Status:** `refuted` · **Source:** Goal 2.2 (knob sensitivity), follow-up to H6


## Claim

H6 found the queue-delay-target knob moves latency but not picture quality, because the encoder had bytes to spare. With the bitrate ceiling dropped from 4000 to 800 kbps, the encoder is hungry, so loosening the knob should now raise picture quality (PSNR) as well as latency.

## Predictions

- `psnr_rises_with_knob_on_every_network`
- `psnr_swing_across_the_knob_is_larger_than_in_h6`
- `p95_latency_rises_with_knob_on_every_network`

## Verdict

| Outcome | Predicate |
|---|---|
| **Supported when all** | <code>psnr_rises_with_knob_on_every_network</code><br><code>psnr_swing_across_the_knob_is_larger_than_in_h6</code> |
| **Refuted when any** | <code>psnr_does_not_rise_with_knob_on_at_least_one_network</code> |
| **Untested when any** | <code>any_cell_failed</code><br><code>required_metric_missing</code> |


## Findings and Limitations

**Findings**

- Starving the encoder did not turn the knob into a quality knob. At the tight 800 kbps ceiling, loosening the knob does not raise picture quality. On the fixed 5 Mbps link PSNR is dead flat (34.2 dB at every knob value); on the capacity-step link (5 Mbps drops to 300 kbps and recovers) it actually falls as the knob loosens (26.4 dB at 10 ms down to about 19 dB at 100 ms and beyond); on the 5G CQI trace it drifts slightly down (31.8 to 30.6 dB). So the queue-delay-target is a latency knob, not a quality knob, at both the loose (H6) and tight (H7) ceilings for this workload.
- The quality numbers are now trustworthy. The decoded_psnr metric was fixed before this run (frame-accurate self-healing alignment), and the values track the bitrate ceiling as they should -- about 32 to 39 dB at the loose 4000 kbps ceiling (H6) versus 19 to 34 dB at the tight 800 kbps ceiling here. The metric clearly separates the two ceilings, which the earlier broken metric (pinned near 10 dB everywhere) could not.
- The robust takeaway across H6 and H7 is one finding, tested at two ceilings with a trustworthy quality axis -- the knob moves latency, not quality, and on the capacity-step link a looser queue-delay-target slightly hurts quality rather than helping (more data buffered through the 300 kbps trough arrives stale or lost).

**Limitations**

- Workload-specific (realmotion-avi, 1280x1024 MJPEG, 10 fps, 60 s) and bitrate-specific (the two ceilings tested). A different clip or ceiling could behave differently.
- The quality axis is PSNR; SSIM or VMAF might surface a smaller effect PSNR misses.
- 3 reps per cell; effects smaller than the run-to-run spread are not resolved.
- Absolute latency carries the aum/veda clock-skew artifact (as in H4/H6); the comparison across knob values within one sweep is unaffected.


## Figures

![Decoded PSNR vs queue-delay-target. Flat or falling on every network — loosening the knob does not raise quality.](results/h7_psnr_by_knob.svg)

*Decoded PSNR vs queue-delay-target. Flat or falling on every network — loosening the knob does not raise quality.*

![p95 frame latency vs queue-delay-target. Latency tracks the knob (the relative trend within a sweep is what matters; the absolute offset carries the aum/veda clock-skew artifact).](results/h7_latency_by_knob.svg)

*p95 frame latency vs queue-delay-target. Latency tracks the knob (the relative trend within a sweep is what matters; the absolute offset carries the aum/veda clock-skew artifact).*


## Tables

### `Decoded PSNR (dB) by queue-delay-target`

| network | 10 ms | 30 ms | 60 ms | 100 ms | 200 ms | 500 ms |
| --- | --- | --- | --- | --- | --- | --- |
| fixed 5 Mbps | 34.18 | 34.19 | 34.19 | 34.18 | 34.19 | 34.17 |
| capacity step (5 Mbps ⇄ 300 kbps) | 26.39 | 26.06 | 25.92 | 19.37 | 19.28 | 19.49 |
| 5G CQI trace | 31.84 | 31.72 | 31.21 | 30.90 | 30.86 | 30.59 |

### `p95 frame latency (ms) by queue-delay-target`

| network | 10 ms | 30 ms | 60 ms | 100 ms | 200 ms | 500 ms |
| --- | --- | --- | --- | --- | --- | --- |
| fixed 5 Mbps | -66.32 | -66.24 | -66.25 | -66.67 | -66.83 | -67.01 |
| capacity step (5 Mbps ⇄ 300 kbps) | -40.32 | 12.52 | 12.34 | -14.19 | 12.22 | -14.20 |
| 5G CQI trace | -46.49 | -31.99 | -28.45 | -27.13 | -30.23 | -26.59 |


## Experimental setup

### `h7-qdt-sweep-static`

queue-delay-target sweep on the static network, tight 800 kbps ceiling.

**Configurations:** `119` (scream qdt=10ms), `120` (scream qdt=30ms), `121` (scream qdt=60ms), `122` (scream qdt=100ms), `123` (scream qdt=200ms), `124` (scream qdt=500ms) · **Reps:** 3

Spec: `specs/experiments/h7-qdt-sweep-static.yaml` · Record: `runs/experiments/h7-qdt-sweep-static.json` · Run: `python3 tools/run_qdt_sweeps.py  # or experiment.py h7-qdt-sweep-static --resume`

**Status:** 18 of 18 runs completed.

### `h7-qdt-sweep-fluct`

queue-delay-target sweep on the fluct network, tight 800 kbps ceiling.

**Configurations:** `125` (scream qdt=10ms), `126` (scream qdt=30ms), `127` (scream qdt=60ms), `128` (scream qdt=100ms), `129` (scream qdt=200ms), `130` (scream qdt=500ms) · **Reps:** 3

Spec: `specs/experiments/h7-qdt-sweep-fluct.yaml` · Record: `runs/experiments/h7-qdt-sweep-fluct.json` · Run: `python3 tools/run_qdt_sweeps.py  # or experiment.py h7-qdt-sweep-fluct --resume`

**Status:** 18 of 18 runs completed.

### `h7-qdt-sweep-5g`

queue-delay-target sweep on the 5g network, tight 800 kbps ceiling.

**Configurations:** `131` (scream qdt=10ms), `132` (scream qdt=30ms), `133` (scream qdt=60ms), `134` (scream qdt=100ms), `135` (scream qdt=200ms), `136` (scream qdt=500ms) · **Reps:** 3

Spec: `specs/experiments/h7-qdt-sweep-5g.yaml` · Record: `runs/experiments/h7-qdt-sweep-5g.json` · Run: `python3 tools/run_qdt_sweeps.py  # or experiment.py h7-qdt-sweep-5g --resume`

**Status:** 18 of 18 runs completed.


## Required metrics

- `frame_count`
- `encoder_target_kbps`
- `wire_bytes`
- `encoded_bitrate`
- `frame_latency`
- `decoder_errors`
- `decoded_psnr`


## Reproducibility

This notebook is generated from `specs/hypotheses/h7.yaml` and `analysis/hypotheses/results/h7_report.json`. To regenerate:

```sh
python3 analysis/hypotheses/build_reports.py
python3 analysis/hypotheses/build_pages.py
python3 analysis/hypotheses/h7_qdt_sweep_tight.py
```

Source: Goal 2.2 (knob sensitivity), follow-up to H6
